# Tugas 2

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    precision_score, recall_score, f1_score
)

In [2]:
raw_df = pd.read_csv('burnout_dataset1.csv')
raw_df['Profil'] = (
    'occupation_' + raw_df['Occupation'].fillna('unknown').astype(str) + ' ' +
    'stress_' + raw_df['Stress_Level'].fillna('unknown').astype(str) + ' ' +
    'sleep_' + raw_df['Sleep_Quality'].fillna('unknown').astype(str) + ' ' +
    'chronic_' + raw_df['Chronic_Stress'].fillna('unknown').astype(str) + ' ' +
    'exercise_' + raw_df['Exercise_Frequency'].fillna('unknown').astype(str) + ' ' +
    'diet_' + raw_df['Healthy_Diet'].fillna('unknown').astype(str) + ' ' +
    'support_' + raw_df['Support_System'].fillna('unknown').astype(str) + ' ' +
    'remote_' + raw_df['Remote_Work'].fillna('unknown').astype(str) + ' ' +
    'alcohol_' + raw_df['Alcohol_Consumption'].fillna('unknown').astype(str) + ' ' +
    'smoking_' + raw_df['Smoking'].fillna('unknown').astype(str) + ' ' +
    raw_df['AI_Wellness_Recommendation'].fillna('').astype(str)
)

df = pd.DataFrame({
    'Labels': (raw_df['Burnout_Risk'] == 'High').astype(int),
    'Profil': raw_df['Profil']
})
df.head()

,Labels,Profil
0,1,occupation_Marketing Manager stress_High sleep...
1,0,occupation_Freelancer stress_Moderate sleep_Po...
2,0,occupation_Accountant stress_Moderate sleep_Av...
3,0,occupation_Chef stress_Low sleep_Good chronic_...
4,0,occupation_Farmer stress_Low sleep_Excellent c...


In [3]:
X = df['Profil'].to_numpy()
y = df['Labels'].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=50
)
print('Jumlah data latih:', len(X_train))
print('Jumlah data uji:', len(X_test))

Jumlah data latih: 40000
Jumlah data uji: 10000


In [4]:
# Multinomial Naive Bayes dengan CountVectorizer dan stop words
bow = CountVectorizer(stop_words='english')
X_train_count = bow.fit_transform(X_train)
X_test_count = bow.transform(X_test)

mnb_count = MultinomialNB()
mnb_count.fit(X_train_count, y_train)
y_train_pred_count = mnb_count.predict(X_train_count)
y_test_pred_count = mnb_count.predict(X_test_count)

print('Jumlah fitur CountVectorizer:', len(bow.get_feature_names_out()))

Jumlah fitur CountVectorizer: 70


In [5]:
acc_train_count = accuracy_score(y_train, y_train_pred_count)
acc_test_count = accuracy_score(y_test, y_test_pred_count)

print('CountVectorizer dengan stop_words=english')
print('Akurasi data train:', acc_train_count)
print('Akurasi data test:', acc_test_count)
print('Confusion Matrix (urutan: Non-High, High):')
print(confusion_matrix(y_test, y_test_pred_count, labels=[0, 1]))
print('Laporan Klasifikasi:')
print(classification_report(y_test, y_test_pred_count, labels=[0, 1], target_names=['Non-High', 'High']))

CountVectorizer dengan stop_words=english
Akurasi data train: 0.918175
Akurasi data test: 0.9185
Confusion Matrix (urutan: Non-High, High):
[[7912  142]
 [ 673 1273]]
Laporan Klasifikasi:
              precision    recall  f1-score   support

    Non-High       0.92      0.98      0.95      8054
        High       0.90      0.65      0.76      1946

    accuracy                           0.92     10000
   macro avg       0.91      0.82      0.85     10000
weighted avg       0.92      0.92      0.91     10000



In [6]:
# Multinomial Naive Bayes dengan TF-IDF dan stop words
tfidf = TfidfVectorizer(stop_words='english')
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

mnb_tfidf = MultinomialNB()
mnb_tfidf.fit(X_train_tfidf, y_train)
y_train_pred_tfidf = mnb_tfidf.predict(X_train_tfidf)
y_test_pred_tfidf = mnb_tfidf.predict(X_test_tfidf)

print('Jumlah fitur TF-IDF:', len(tfidf.get_feature_names_out()))

Jumlah fitur TF-IDF: 70


In [7]:
acc_train_tfidf = accuracy_score(y_train, y_train_pred_tfidf)
acc_test_tfidf = accuracy_score(y_test, y_test_pred_tfidf)

print('TF-IDF dengan stop_words=english')
print('Akurasi data train:', acc_train_tfidf)
print('Akurasi data test:', acc_test_tfidf)
print('Confusion Matrix (urutan: Non-High, High):')
print(confusion_matrix(y_test, y_test_pred_tfidf, labels=[0, 1]))
print('Laporan Klasifikasi:')
print(classification_report(y_test, y_test_pred_tfidf, labels=[0, 1], target_names=['Non-High', 'High']))

TF-IDF dengan stop_words=english
Akurasi data train: 0.915375
Akurasi data test: 0.9166
Confusion Matrix (urutan: Non-High, High):
[[7938  116]
 [ 718 1228]]
Laporan Klasifikasi:
              precision    recall  f1-score   support

    Non-High       0.92      0.99      0.95      8054
        High       0.91      0.63      0.75      1946

    accuracy                           0.92     10000
   macro avg       0.92      0.81      0.85     10000
weighted avg       0.92      0.92      0.91     10000



In [8]:
hasil_evaluasi = pd.DataFrame({
    'Akurasi train': [acc_train_count, acc_train_tfidf],
    'Akurasi test': [acc_test_count, acc_test_tfidf],
    'Precision High': [
        precision_score(y_test, y_test_pred_count),
        precision_score(y_test, y_test_pred_tfidf)
    ],
    'Recall High': [
        recall_score(y_test, y_test_pred_count),
        recall_score(y_test, y_test_pred_tfidf)
    ],
    'F1-score High': [
        f1_score(y_test, y_test_pred_count),
        f1_score(y_test, y_test_pred_tfidf)
    ]
}, index=['CountVectorizer + stop_words', 'TF-IDF + stop_words'])
hasil_evaluasi

,Akurasi train,Akurasi test,Precision High,Recall High,F1-score High
CountVectorizer + stop_words,0.918175,0.9185,0.899647,0.654162,0.757513
TF-IDF + stop_words,0.915375,0.9166,0.913690,0.631038,0.746505


Kedua model menggunakan Multinomial Naive Bayes dengan parameter bawaan, stop words bahasa Inggris, dan pembagian data yang sama: 80% data latih serta 20% data uji dengan random_state=50. Vectorizer hanya dilatih pada data latih.

CountVectorizer menghasilkan akurasi test 91,85%, precision High 89,96%, recall High 65,42%, dan F1-score High 75,75%. TF-IDF menghasilkan akurasi test 91,66%, precision High 91,37%, recall High 63,10%, dan F1-score High 74,65%. Pada kedua model terlihat bahwa TF-IDF memberikan precision sedikit lebih tinggi untuk kelas High, namun melewatkan lebih banyak kasus risiko burnout tinggi sebagai Non-High dibandingkan CountVectorizer.

Pada percobaan ini, fitur terbaik berdasarkan F1-score kelas High adalah CountVectorizer + stop_words. F1-score dipakai sebagai dasar kesimpulan karena mempertimbangkan keseimbangan precision dan recall pada kelas risiko burnout tinggi (High) yang jumlahnya lebih sedikit dibandingkan kelas Non-High.